# Exploração gráfica percentual dos dados processados do INCT SiMAI

Gráficos sobre `dados/processados/inct_limpo.json`. A parte textual e numérica está em `exploracao_dados_processados.ipynb`.

In [1]:
import json
from pathlib import Path

import pandas as pd

CAMINHO = Path('..', '..') / 'dados' / 'processados' / 'inct_limpo.json'

with open(CAMINHO, encoding='utf-8') as f:
    registros = json.load(f)

df = pd.DataFrame(registros)

print(df.shape)

(6674, 7)


## Gráficos para inspecionar as inconsistências

Cada gráfico mostra dois bairros, em cores diferentes. O eixo vertical é sempre um percentual, para que bairros de tamanhos muito diferentes fiquem comparáveis: **% de prédios afetados** (`percentage`) e **% da população afetada** (`affected_population / population`). A linha pontilhada marca o nível recorde de maio de 2024 (5,35 m).

In [2]:
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BAIRROS = sorted(df['bairro'].unique())
CORES = ['#1f77b4', '#d62728']

def graficos_pares(valor_y, titulo_y):
    # Cada gráfico mostra dois bairros em cores diferentes; dois gráficos por linha
    pares = [BAIRROS[i:i + 2] for i in range(0, len(BAIRROS), 2)]
    titulos = []
    for par in pares:
        partes = [f"<span style='color:{cor}'>{b}</span>" for b, cor in zip(par, CORES)]
        titulos.append(' × '.join(partes))
    fig = make_subplots(rows=math.ceil(len(pares) / 2), cols=2, subplot_titles=titulos,
                        vertical_spacing=min(0.06, 0.25 / (math.ceil(len(pares) / 2) - 1)), horizontal_spacing=0.08)
    for k, par in enumerate(pares):
        linha, coluna = k // 2 + 1, k % 2 + 1
        for bairro, cor in zip(par, CORES):
            s = df[df['bairro'] == bairro].sort_values('nivel_m')
            fig.add_scatter(x=s['nivel_m'], y=valor_y(s), mode='lines+markers', name=bairro,
                            showlegend=False, line=dict(color=cor), marker=dict(size=4, color=cor),
                            row=linha, col=coluna)
        fig.add_vline(x=5.35, line_dash='dot', row=linha, col=coluna)
    fig.update_yaxes(range=[0, 100], title_text=titulo_y)
    fig.update_xaxes(title_text='Nível do Guaíba (m)')
    fig.update_layout(title=titulo_y, height=380 * math.ceil(len(pares) / 2) + 120,
                      margin=dict(t=100, b=40))
    fig.show()

### % de prédios afetados por nível, dois bairros por gráfico

In [3]:
graficos_pares(lambda s: s['percentage'], '% de prédios afetados')

### % da população afetada por nível, dois bairros por gráfico

In [4]:
graficos_pares(lambda s: s['affected_population'] / s['population'] * 100, '% da população afetada')

### % da população afetada na cidade por nível

In [5]:
import plotly.express as px

cidade = df.groupby('nivel_m')[['affected_population', 'population']].sum().reset_index()
cidade['% população afetada'] = cidade['affected_population'] / cidade['population'] * 100
fig = px.line(cidade, x='nivel_m', y='% população afetada', markers=True,
              labels={'nivel_m': 'Nível do Guaíba (m)'},
              title='% da população da cidade afetada por nível')
fig.add_vline(x=5.35, line_dash='dot', annotation_text='maio/2024')
fig.update_yaxes(range=[0, 100])
fig.show()